In [11]:
import polars as pl

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, chi2

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

import numpy as np

In [8]:
file = "/content/drive/MyDrive/NLP/train_arcaico_moderno.csv"
#file = "/content/drive/MyDrive/NLP/train_complexo_simples.csv"
#file = "/content/drive/MyDrive/NLP/train_literal_dinamico.csv"

csv = pl.read_csv(source=file, separator=";").filter(pl.col("text").is_not_null())

label_text = list(set(csv["style"]))

labels = [0 if style == label_text[0] else 1 for style in csv["style"]]
texts = [text for text in csv['text']]

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.3, random_state=42)

In [12]:
pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(1,5), min_df=3))
    ])),
    ("kbest", SelectKBest(chi2, k=22000)),
    ("clf", LinearSVC(C=0.9))
])

model = pipeline.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


scores = cross_val_score(pipeline, texts, labels, cv=10, scoring='accuracy')
print("Acurácias por fold:", scores)
print("Acurácia média:", np.mean(scores))


              precision    recall  f1-score   support

           0       0.89      0.88      0.89      5549
           1       0.88      0.89      0.89      5517

    accuracy                           0.89     11066
   macro avg       0.89      0.89      0.89     11066
weighted avg       0.89      0.89      0.89     11066

Acurácias por fold: [0.89997289 0.89671998 0.90132827 0.90485226 0.89723427 0.90347072
 0.89777657 0.89831887 0.89804772 0.89886117]
Acurácia média: 0.8996582734976293
